In [0]:
import importlib
from config.config import *

from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
silver_tables = dbutils.fs.ls(silver_path)
for table in silver_tables:
  print(table.name)

In [0]:
def inspect_silver_tables(path):

    entries = dbutils.fs.ls(path)

    for entry in entries:

        if entry.isDir():

            table_name = entry.name.rstrip("/")

            try:
                df = spark.read.format("delta").load(entry.path)

                print("\n" + "=" * 80)
                print(f"SILVER TABLE: {table_name}")
                print("=" * 80)

                print("\nSCHEMA:")
                df.printSchema()

                print("\nROW COUNT:")
                print(df.count())

                print("\nSAMPLE DATA:")
                display(df)

            except Exception as e:
                print(f"Could not inspect {table_name}: {e}")


inspect_silver_tables(silver_path)

In [0]:
def inspect_silver_columns(path):

    entries = dbutils.fs.ls(path)

    for entry in entries:

        if entry.isDir():

            table_name = entry.name.rstrip("/")

            try:
                df = spark.read.format("delta").load(entry.path)

                print("\n" + "=" * 80)
                print(f"SILVER TABLE: {table_name}")
                print("=" * 80)

                for field in df.schema.fields:
                    print(f"{field.name:<40} {field.dataType}")

            except Exception as e:
                print(f"Could not inspect {table_name}: {e}")


inspect_silver_columns(silver_path)

In [0]:
from pyspark.sql.functions import col, countDistinct, collect_set

tables = [
    "estat_tour_occ_arm",
    "estat_tour_occ_mnor",
    "estat_tour_occ_nim",
    "un_tourism_accommodation_hotels",
    "un_tourism_inbound_arrivals",
    "world_bank_international_arrivals"
]

# Columns to exclude from distinct-value inspection
exclude_columns = {
    "period",
    "year",
    "tourism_value",
    "occupancy_rate",
    "nights_spent",
    "hotel_capacity",
    "arrivals"
}

for table in tables:

    print("\n" + "=" * 80)
    print(f"SILVER TABLE: {table}")
    print("=" * 80)

    df = spark.read.format("delta").load(f"{silver_path}/{table}")

    # Inspect categorical and identifier columns
    columns_to_check = [
        field.name
        for field in df.schema.fields
        if field.name not in exclude_columns
    ]

    for column_name in columns_to_check:

        print(f"\nColumn: {column_name}")

        result = (
            df.select(col(column_name))
            .distinct()
            .orderBy(col(column_name))
        )

        print(f"Unique values: {result.count()}")

        result.show(200, truncate=False)